# G3 LightGBM 시드 평균 — 팀 ②번 요청 독립 실험

팀의 `10_lgbm_g3_3model_equal_ensemble.ipynb`에서 **피처·모델·그룹 CV·1:1:1 앙상블 설정**을 가져왔습니다. 33번 실험은 사용하지 않습니다. 원본 팀 파일은 변경하지 않습니다.

**주의:** 이 원본 10번 구성과 팀장이 말한 *11번 제출 구성*이 정확히 같은지는 아직 증명되지 않았습니다. 따라서 결과를 공식 11번 OOF로 부르지 않습니다. 개발 CV +0.003 조건 충족 시에만 test CSV 생성 옵션을 열되, 제출 전 공식 11번 구성과의 일치 여부를 별도 확인하세요. test의 분포나 통계는 출력하지 않습니다.

**실행:** 첫 번째 사전 점검 셀부터 순서대로 실행하세요. 데이터 경로가 다르면 데이터 로딩 셀의 `DATA_DIR_OVERRIDE`만 설정합니다. 기존 팀 파일은 변경하지 않습니다.


In [5]:
# 실행 전 환경 및 데이터 파일 점검 — 기존 팀 공통 파일 수정 없음
from pathlib import Path
import sys

PROJECT_ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                     if (p / "src" / "config.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("프로젝트의 src/config.py를 찾을 수 없습니다. AH07 프로젝트 안에서 노트북을 실행하세요.")

print("프로젝트:", PROJECT_ROOT)
print("Python:", sys.executable)
try:
    import numpy as np
    import scipy
    from scipy.sparse import csr_matrix
    import sklearn
    import pandas as pd
    import lightgbm
    import xgboost
except (ImportError, AttributeError) as exc:
    raise RuntimeError(
        "현재 노트북 커널에서 ML 패키지 import에 실패했습니다. VS Code에서 프로젝트 .venv 커널을 "
        "선택하고 Restart Kernel을 실행한 뒤 처음부터 다시 실행하세요. "
        "실행 중 sys.modules에서 numpy/scipy를 삭제하지 마세요."
    ) from exc
print("라이브러리:", "numpy", np.__version__, "scipy", scipy.__version__,
      "sklearn", sklearn.__version__, "lightgbm", lightgbm.__version__,
      "xgboost", xgboost.__version__)
assert csr_matrix([[1, 0], [0, 1]]).shape == (2, 2)
print("라이브러리 사전 점검 통과")


프로젝트: C:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype
Python: c:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype\.venv\Scripts\python.exe
라이브러리: numpy 1.26.4 scipy 1.13.1 sklearn 1.5.2 lightgbm 4.7.0 xgboost 3.2.0
라이브러리 사전 점검 통과


In [6]:
from pathlib import Path
import sys, json, warnings
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

start = Path.cwd().resolve()
ROOT = next((p for p in (start, *start.parents) if (p/'src'/'config.py').is_file() and (p/'scripts'/'fixed_holdout_manifest.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('src/config.py가 있는 프로젝트 폴더에서 실행하세요.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# 노트북의 현재 작업 폴더와 무관하게 프로젝트 스크립트 로딩
import importlib.util
manifest_path = ROOT / 'scripts' / 'fixed_holdout_manifest.py'
spec = importlib.util.spec_from_file_location('fixed_holdout_manifest', manifest_path)
holdout_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(holdout_module)
assert holdout_module.ROOT.resolve() == ROOT.resolve(), '스크립트의 ROOT 경로를 확인하세요.'
from src.config import DATA_DIR, SEED, set_seed
from src.data_loader import load_data, split_xy, encode_target
from src.preprocessing import binarize_mutations
from src.validation import get_folds, run_cv
from src.evaluation import macro_f1, save_predictions, log_result
from src.features import mutation_count, mutation_type_counts, amino_acid_pair_counts, gene_truncation_flags

set_seed()
RUN_FULL_CV = False    # 팀장 승인 후 True: 전체 OOF/Test 생성 (기존 정상 파일 재사용)
SAVE_ARTIFACTS = False # True: 3개 모델 + 앙상블 예측 파일 저장
SAVE_SUBMISSION = False # 팀 규칙: 사용하지 않음
print('ROOT', ROOT, '| XGBoost', __import__('xgboost').__version__)
print('홀드아웃 스크립트:', manifest_path)

ROOT C:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype | XGBoost 3.2.0
홀드아웃 스크립트: C:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype\scripts\fixed_holdout_manifest.py


In [7]:
def select_xgb_device():
    try:
        probe = XGBClassifier(n_estimators=1, max_depth=1, tree_method='hist', device='cuda',
                              objective='binary:logistic', eval_metric='logloss', verbosity=0)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            probe.fit(np.array([[0.],[1.],[2.],[3.]],dtype=np.float32), np.array([0,0,1,1]))
        actual = json.loads(probe.get_booster().save_config())['learner']['generic_param']['device']
        if str(actual).startswith('cuda'):
            return 'cuda'
    except Exception as exc:
        print('CUDA 확인 실패:', repr(exc))
    return 'cpu'

DEVICE = select_xgb_device()
print('XGBoost device:', DEVICE)

XGBoost device: cuda


In [8]:
# train.csv와 test.csv가 같은 폴더에 있는지 확인합니다.
# 자동 탐색에서 여러 데이터셋이 발견되면 정확한 원본을 직접 지정해야 합니다.
# 예: DATA_DIR_OVERRIDE = Path(r"C:\Users\user\Desktop\AI_Projects\ah07-holdout-fix\data")
DATA_DIR_OVERRIDE = None

from pathlib import Path


def resolve_data_dir(project_root, override=None):
    project_root = Path(project_root).resolve()
    if override is not None:
        chosen = Path(override).expanduser().resolve()
        missing = [name for name in ("train.csv", "test.csv") if not (chosen / name).is_file()]
        if missing:
            raise FileNotFoundError(f"지정한 데이터 폴더 {chosen} 에 {', '.join(missing)} 파일이 없습니다.")
        return chosen

    # 1) 팀 공식 data 폴더를 우선합니다.
    official = project_root / "data"
    if (official / "train.csv").is_file() and (official / "test.csv").is_file():
        return official

    # 2) 기존 AI_Projects 폴더와 프로젝트 내부의 데이터 폴더를 검색합니다.
    #    .venv/.git 등 환경 폴더는 탐색하지 않습니다.
    roots = [project_root, project_root.parent]
    found = set()
    skip = {".venv", ".git", "node_modules", "__pycache__", ".ipynb_checkpoints"}
    import os
    for root in roots:
        for current, dirs, files in os.walk(root):
            dirs[:] = [d for d in dirs if d not in skip]
            if "train.csv" in files and "test.csv" in files:
                found.add(Path(current).resolve())
    matches = sorted(found, key=str)
    if len(matches) == 1:
        return matches[0]
    if not matches:
        raise FileNotFoundError(
            "train.csv와 test.csv를 같은 폴더에서 찾지 못했습니다. "
            "원본 데이터가 있는 폴더를 DATA_DIR_OVERRIDE에 지정하거나 "
            f"{official} 폴더에 원본 파일을 배치하세요. "
            "데이터 파일을 임의 생성하거나 팀 공통 파일을 변경하지 않습니다."
        )
    raise RuntimeError(
        "train.csv/test.csv가 들어 있는 폴더가 여러 개 있습니다. "
        "잘못된 실험 데이터 사용을 방지하기 위해 자동 선택하지 않습니다. "
        "DATA_DIR_OVERRIDE에 사용할 폴더를 지정하세요:\n" +
        "\n".join(str(p) for p in matches)
    )

DATA_DIR = resolve_data_dir(ROOT, DATA_DIR_OVERRIDE)
print("사용할 데이터 폴더:", DATA_DIR)
train, test = load_data(DATA_DIR)

X_raw, y_raw = split_xy(train)
y, le = encode_target(y_raw)
X_test_raw = test[X_raw.columns].copy()
assert X_raw.columns.equals(X_test_raw.columns)
assert X_raw.index.equals(pd.RangeIndex(len(X_raw)))
assert len(le.classes_) == 26
# 공통 홀드아웃 먼저 읽기: CV는 개발 데이터에서만 실행합니다.
dev_idx, hold_idx = holdout_module.verify()
assert len(dev_idx) + len(hold_idx) == len(y)
assert not np.intersect1d(dev_idx, hold_idx).size
X_dev_raw = X_raw.iloc[dev_idx].reset_index(drop=True)
y_dev = np.asarray(y)[dev_idx]
folds = get_folds(y_dev, X=X_dev_raw, classes=le.classes_)  # 팀 함수 변경 금지
print('train', X_raw.shape, 'test', X_test_raw.shape, 'classes', len(le.classes_))

def make_E(X):
    # 시윤님 02_features E와 동일한 피처 순서/변환
    parts = [binarize_mutations(X), np.log1p(mutation_count(X)),
             np.log1p(mutation_type_counts(X)), np.log1p(amino_acid_pair_counts(X)),
             gene_truncation_flags(X)]
    out = pd.concat(parts, axis=1).astype(np.float32)
    assert not out.columns.has_duplicates and out.index.equals(X.index)
    assert np.isfinite(out.to_numpy()).all()
    return out

X_E = make_E(X_raw)
X_test_E = make_E(X_test_raw)
assert X_E.columns.equals(X_test_E.columns)
assert X_E.shape[1] == 9174, f'E 피처 개수 불일치: {X_E.shape[1]}'
print('E', X_E.shape, X_test_E.shape)
X_dev_E = X_E.iloc[dev_idx].reset_index(drop=True)


사용할 데이터 폴더: C:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype\data
VERIFIED: dev=4960, holdout=1241; same IDs and row signatures
train (6201, 4384) test (2546, 4384) classes 26
E (6201, 9174) (2546, 9174)


## 1. 기존 3번째 모델: E + Position-bin LR (그대로 유지)

In [9]:
# 시윤님 03_hotspot_bins.ipynb의 E+Position-bin 함수 재현 (핫스팟 미사용)
import re
from collections import Counter
from scipy.sparse import hstack
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

def make_features_E_sparse(X):
    return csr_matrix(make_E(X).to_numpy(dtype=np.float32))

def extract_position(variant):
    match = re.search(r"\d+", variant)
    return int(match.group()) if match else None
# Position-bin 후보 생성 함수

from collections import defaultdict

def extract_position_bins(X, bin_size=50):
    rows = []
    cols = []
    gene_bins = set()

    for row_idx, (_, row) in enumerate(X.iterrows()):
        patient_bins = set()

        for gene, value in row.items():
            if gene == "ID":
                continue

            if pd.isna(value) or str(value) == "WT":
                continue

            for variant in str(value).split():
                pos = extract_position(variant)

                if pos is not None:
                    feature = f"{gene}_bin_{pos // bin_size}"
                    patient_bins.add(feature)

        for feature in patient_bins:
            rows.append(row_idx)
            cols.append(feature)
            gene_bins.add(feature)

    return rows, cols, gene_bins


# Fold별 Position-bin 변환기 구현

from sklearn.base import BaseEstimator, TransformerMixin
from scipy.sparse import csr_matrix
from collections import Counter

class PositionBinTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, bin_size=50, min_patients=5):
        self.bin_size = bin_size
        self.min_patients = min_patients

    def fit(self, X, y=None):
        _, cols, _ = extract_position_bins(
            X, bin_size=self.bin_size
        )

        counts = Counter(cols)

        self.selected_bins_ = sorted(
            feature
            for feature, count in counts.items()
            if count >= self.min_patients
        )

        self.bin_to_idx_ = {
            feature: idx
            for idx, feature in enumerate(self.selected_bins_)
        }

        return self

    def transform(self, X):
        rows, cols, _ = extract_position_bins(
            X, bin_size=self.bin_size
        )

        selected_rows = []
        selected_cols = []

        for row_idx, feature in zip(rows, cols):
            col_idx = self.bin_to_idx_.get(feature)

            if col_idx is not None:
                selected_rows.append(row_idx)
                selected_cols.append(col_idx)

        data = np.ones(len(selected_rows), dtype=np.float32)

        return csr_matrix(
            (data, (selected_rows, selected_cols)),
            shape=(len(X), len(self.selected_bins_)),
            dtype=np.float32
        )# 피처 구성 E + Position-bin CV

from scipy.sparse import hstack

class FeaturesEWithPositionBins(BaseEstimator, TransformerMixin):
    def __init__(self, bin_size=50, min_patients=5):
        self.bin_size = bin_size
        self.min_patients = min_patients

    def fit(self, X, y=None):
        self.position_transformer_ = PositionBinTransformer(
            bin_size=self.bin_size,
            min_patients=self.min_patients
        )
        self.position_transformer_.fit(X)
        return self

    def transform(self, X):
        # 기존 피처 구성 E
        X_base = make_features_E_sparse(X)

        # Fold 학습 데이터에서 선택한 Position-bin
        X_position = self.position_transformer_.transform(X)

        return hstack(
            [X_base, X_position],
            format="csr",
            dtype=np.float32
        )


pipeline_E_position_LR = Pipeline([
    ("features", FeaturesEWithPositionBins(
        bin_size=50,
        min_patients=5
    )),
    ("model", LogisticRegression(
        C=0.1,
        class_weight="balanced",
        max_iter=3000,
        solver="lbfgs"
    ))
])



## 2. LightGBM g3 변환기: Hotspot×3, Position-bin, Exact-position

In [10]:
# 유전자별 Hotspot 아미노산 위치
HOTSPOTS = {
    "IDH1": ["R132"],
    "IDH2": ["R140", "R172"],
    "BRAF": ["V600"],
    "HRAS": ["Q61", "G13", "G12"],
    "KIT": ["D816", "N822", "K642"],
    "CTNNB1": ["S33", "S37", "S45", "T41", "G34", "D32"],
    "U2AF1": ["S34"],
    "PIK3CA": ["E545", "H1047", "E542"],
    "EGFR": ["L858", "G598", "A289"],
    "RET": ["M918"],
    "RAC1": ["P29"],
    "SPOP": ["F133", "W131", "F102", "Y87"],
    "EZH2": ["Y641", "Y646"],
    "TP53": ["R175", "R248", "R273"],
    "PTEN": ["R130"],
}


assert sum(map(len,HOTSPOTS.values())) == 35

In [11]:
def make_hotspot_features(X):
    hotspot_data = {}

    # 일반 Hotspot
    for gene, positions in HOTSPOTS.items():
        if gene not in X.columns:
            raise ValueError(f"유전자 컬럼 없음: {gene}")

        tokens = X[gene].fillna("WT").astype(str).str.split()

        for position in positions:
            col_name = f"hotspot_{gene}_{position}"

            hotspot_data[col_name] = tokens.map(
                lambda variants: int(
                    any(v.startswith(position) for v in variants)
                )
            ).astype(np.int8)

    # NPM1 frameshift
    if "NPM1" not in X.columns:
        raise ValueError("유전자 컬럼 없음: NPM1")

    hotspot_data["hotspot_NPM1_fs"] = (
        X["NPM1"]
        .fillna("WT")
        .astype(str)
        .str.contains("fs", case=False, regex=False)
        .astype(np.int8)
    )

    return pd.DataFrame(hotspot_data, index=X.index)




In [12]:
from lightgbm import LGBMClassifier
from sklearn.base import BaseEstimator, TransformerMixin
from collections import Counter

# 변이 토큰의 시작에 있는 단일 영문자 + 숫자만 인식: V600E -> V600, R132H -> R132.
# 468_469LG처럼 숫자로 시작하거나 복잡한 표기는 의도적으로 제외.
EXACT_POSITION_RE = re.compile(r'^([A-Za-z])(\d+)')

def patient_exact_positions(X):
    result = []
    for row in X.itertuples(index=False, name=None):
        found = set()
        for gene, value in zip(X.columns, row):
            if gene in ('ID', 'SUBCLASS') or pd.isna(value) or str(value).strip().upper() == 'WT':
                continue
            for token in str(value).split():
                match = EXACT_POSITION_RE.match(token)
                if match:
                    found.add(f'{gene}_{match.group(1).upper()}{match.group(2)}')
        result.append(found)
    return result

class ExactPositionTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, min_patients=5):
        self.min_patients = min_patients

    def fit(self, X, y=None):
        counts = Counter(key for keys in patient_exact_positions(X) for key in keys)
        self.selected_ = sorted(k for k,v in counts.items() if v >= self.min_patients)
        self.mapping_ = {key: i for i,key in enumerate(self.selected_)}
        return self

    def transform(self, X):
        if not hasattr(self, 'mapping_'):
            raise RuntimeError('ExactPositionTransformer.fit()을 먼저 실행하세요')
        rows, cols = [], []
        for i, keys in enumerate(patient_exact_positions(X)):
            for key in keys:
                j = self.mapping_.get(key)
                if j is not None:
                    rows.append(i); cols.append(j)
        return csr_matrix((np.ones(len(rows),dtype=np.float32),(rows,cols)),
                          shape=(len(X),len(self.selected_)),dtype=np.float32)

class FeaturesEWithExactPositions(BaseEstimator, TransformerMixin):
    def __init__(self, min_patients=5):
        self.min_patients = min_patients
    def fit(self, X, y=None):
        self.exact_ = ExactPositionTransformer(self.min_patients).fit(X)
        return self
    def transform(self, X):
        return hstack([make_features_E_sparse(X), self.exact_.transform(X)],
                      format='csr',dtype=np.float32)



In [13]:
class FeaturesEHotspot3BinExact(BaseEstimator, TransformerMixin):
    def __init__(self, bin_size=50, min_patients=5, hotspot_multiplier=3):
        self.bin_size=bin_size
        self.min_patients=min_patients
        self.hotspot_multiplier=hotspot_multiplier

    def fit(self, X, y=None):
        # 선택은 반드시 이 fit에 전달된 학습 Fold 데이터에서만 수행
        self.position_transformer_ = PositionBinTransformer(
            bin_size=self.bin_size, min_patients=self.min_patients).fit(X)
        self.exact_transformer_ = ExactPositionTransformer(
            min_patients=self.min_patients).fit(X)
        return self

    def transform(self, X):
        base = make_features_E_sparse(X)
        hotspot = csr_matrix(make_hotspot_features(X).to_numpy(dtype=np.float32)
                             * self.hotspot_multiplier)
        bins = self.position_transformer_.transform(X)
        exact = self.exact_transformer_.transform(X)
        return hstack([base, hotspot, bins, exact], format='csr', dtype=np.float32)

G3_PARAMS = dict(n_estimators=800, learning_rate=0.02, num_leaves=15,
                 min_child_samples=5, colsample_bytree=0.2, subsample=0.8,
                 subsample_freq=1, class_weight='balanced', random_state=42,
                 verbosity=-1, n_jobs=-1)
lgbm_g3 = Pipeline([
    ('features', FeaturesEHotspot3BinExact(bin_size=50, min_patients=5, hotspot_multiplier=3)),
    ('model', LGBMClassifier(**G3_PARAMS))
])
_demo = pd.DataFrame({'BRAF':['V600E','V600K','WT','V600E','V600E','V600E'], 'IDH1':['WT']*6})
assert ExactPositionTransformer(5).fit(_demo).selected_ == ['BRAF_V600']
print('g3 피처 변환기 준비 완료')

g3 피처 변환기 준비 완료


## 개발 CV: 시드 평균과 기존 g3 비교 (홀드아웃 미사용)

In [14]:
from lightgbm import __version__ as LIGHTGBM_VERSION
from sklearn.metrics import precision_recall_fscore_support
from time import perf_counter

SEEDS = [42, 7, 21]  # 3개 기본, 최대 5개로 확장 가능
# 5개: [42, 7, 21, 123, 2026]
assert 3 <= len(SEEDS) <= 5 and len(SEEDS) == len(set(SEEDS))
K = len(le.classes_)
LABELS = np.arange(K)
OUT = ROOT / 'experiments' / 'g3_seed_average_team_02'
OUT.mkdir(parents=True, exist_ok=True)

common = dict(n_estimators=250, max_depth=3, learning_rate=0.05,
              subsample=0.8, colsample_bytree=0.8,
              objective='multi:softprob', num_class=K, eval_metric='mlogloss',
              tree_method='hist', device=DEVICE, random_state=SEED, n_jobs=-1)
xgb = XGBClassifier(**common)

def weight_array(y_part):
    counts = np.bincount(np.asarray(y_part, dtype=int), minlength=K)
    if np.any(counts == 0):
        raise ValueError('학습 fold에 누락된 클래스가 있습니다')
    w = 1.0 / np.sqrt(counts[np.asarray(y_part, dtype=int)])
    return (w / w.mean()).astype(np.float32)

def weighted_fit_params(y_part):
    return {'sample_weight': weight_array(y_part)}

def checked(p, rows):
    p = np.asarray(p, dtype=np.float64)
    if p.shape != (rows, K) or not np.isfinite(p).all() or not np.allclose(p.sum(1), 1, atol=1e-4):
        raise ValueError(f'확률 행렬 불일치: {p.shape}, expected {(rows,K)}')
    return p

def score(y_true, p):
    return float(f1_score(y_true, p.argmax(1), labels=LABELS, average='macro', zero_division=0))

def folds_score(y_true, p, cv_folds):
    return np.array([score(y_true[va], p[va]) for _, va in cv_folds])

start_time = perf_counter()
px = checked(run_cv(xgb, X_dev_E, y_dev, folds=folds,
                    fit_params_fn=weighted_fit_params)['oof_proba'], len(y_dev))
pp = checked(run_cv(pipeline_E_position_LR, X_dev_raw, y_dev,
                    folds=folds)['oof_proba'], len(y_dev))
seed_probs = []
seed_rows = []
for seed in SEEDS:
    model = clone(lgbm_g3)
    model.set_params(model__random_state=seed)
    p = checked(run_cv(model, X_dev_raw, y_dev, folds=folds)['oof_proba'], len(y_dev))
    seed_probs.append(p)
    seed_rows.append({'seed': seed, 'g3_dev_oof_macro_f1': score(y_dev, p)})
    print('g3 seed', seed, 'OOF', seed_rows[-1]['g3_dev_oof_macro_f1'])

pg_base = seed_probs[0]
pg_mean = checked(np.mean(seed_probs, axis=0), len(y_dev))
ens_base = checked((px + pp + pg_base) / 3, len(y_dev))
ens_mean = checked((px + pp + pg_mean) / 3, len(y_dev))
base_score = score(y_dev, ens_base)
mean_score = score(y_dev, ens_mean)
delta = mean_score - base_score
print('g3 단독 baseline/mean:', score(y_dev,pg_base), score(y_dev,pg_mean))
print('3모델 앙상블 baseline/mean/delta:', base_score, mean_score, f'{delta:+.6f}')
print('경과 시간(초):', round(perf_counter()-start_time,1))
print('LightGBM:', LIGHTGBM_VERSION)


c:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype\.venv\lib\site-packages\xgboost\core.py:751: UserWarning: [01:51:02] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


fold 1: Macro F1 = 0.4459
fold 2: Macro F1 = 0.4950
fold 3: Macro F1 = 0.4553
fold 4: Macro F1 = 0.5001
fold 5: Macro F1 = 0.4821
CV Macro F1 = 0.4757 (std 0.0215)
fold 1: Macro F1 = 0.4803
fold 2: Macro F1 = 0.5045
fold 3: Macro F1 = 0.4839
fold 4: Macro F1 = 0.4897
fold 5: Macro F1 = 0.4838
CV Macro F1 = 0.4885 (std 0.0086)
fold 1: Macro F1 = 0.4806
fold 2: Macro F1 = 0.5011
fold 3: Macro F1 = 0.4725
fold 4: Macro F1 = 0.5064
fold 5: Macro F1 = 0.4958
CV Macro F1 = 0.4913 (std 0.0128)
g3 seed 42 OOF 0.4957689875250478
fold 1: Macro F1 = 0.4821
fold 2: Macro F1 = 0.4821
fold 3: Macro F1 = 0.4746
fold 4: Macro F1 = 0.5048
fold 5: Macro F1 = 0.4976
CV Macro F1 = 0.4882 (std 0.0112)
g3 seed 7 OOF 0.4915494910631383
fold 1: Macro F1 = 0.4651
fold 2: Macro F1 = 0.4859
fold 3: Macro F1 = 0.4665
fold 4: Macro F1 = 0.5158
fold 5: Macro F1 = 0.5039
CV Macro F1 = 0.4874 (std 0.0201)
g3 seed 21 OOF 0.49032521238324794
g3 단독 baseline/mean: 0.4957689875250478 0.49415798644583536
3모델 앙상블 baseline/m

In [15]:
fold_base = folds_score(y_dev, ens_base, folds)
fold_mean = folds_score(y_dev, ens_mean, folds)
fold_table = pd.DataFrame({'fold': range(1,len(folds)+1), 'baseline':fold_base,
                           'seed_mean':fold_mean, 'delta':fold_mean-fold_base})
print(fold_table.to_string(index=False))
print('개선 fold 수:',int((fold_table.delta>0).sum()), '/',len(fold_table))

_, _, f1_base, support = precision_recall_fscore_support(y_dev,ens_base.argmax(1),labels=LABELS,zero_division=0)
_, _, f1_mean, _ = precision_recall_fscore_support(y_dev,ens_mean.argmax(1),labels=LABELS,zero_division=0)
class_table = pd.DataFrame({'cancer':le.classes_.astype(str),'support':support,
 'baseline_f1':f1_base,'seed_mean_f1':f1_mean,'delta':f1_mean-f1_base,
 'baseline_pred_count':np.bincount(ens_base.argmax(1),minlength=K),
 'seed_mean_pred_count':np.bincount(ens_mean.argmax(1),minlength=K)})
print(class_table.to_string(index=False))
pd.DataFrame(seed_rows).to_csv(OUT/'seed_scores.csv',index=False,encoding='utf-8-sig')
fold_table.to_csv(OUT/'fold_scores.csv',index=False,encoding='utf-8-sig')
class_table.to_csv(OUT/'class_scores.csv',index=False,encoding='utf-8-sig')
np.savez_compressed(OUT/'dev_oof_comparison.npz',y_dev=y_dev, g3_base=pg_base,g3_seed_mean=pg_mean,
                   ensemble_base=ens_base,ensemble_seed_mean=ens_mean,dev_idx=dev_idx,seeds=SEEDS)
PASS = bool(delta >= 0.003)
print('개발 OOF +0.003 기준:', '통과' if PASS else '미통과')
print('결과 경로:', OUT)


 fold  baseline  seed_mean     delta
    1  0.477129   0.482695  0.005567
    2  0.516605   0.516388 -0.000217
    3  0.489124   0.490258  0.001134
    4  0.505277   0.510108  0.004831
    5  0.501731   0.503133  0.001402
개선 fold 수: 4 / 5
cancer  support  baseline_f1  seed_mean_f1     delta  baseline_pred_count  seed_mean_pred_count
   ACC       58     0.886792      0.886792  0.000000                   48                    48
  BLCA       84     0.510345      0.510345  0.000000                   61                    61
  BRCA      629     0.538951      0.537125 -0.001826                  629                   637
  CESC      124     0.387931      0.403433  0.015502                  108                   109
  COAD      179     0.772455      0.774775  0.002320                  155                   154
  DLBC       31     0.228571      0.228571  0.000000                    4                     4
GBMLGG      369     0.405360      0.417362  0.012002                  228                

## Test 예측 — 기준 통과 시에만 수동 활성화

`MAKE_TEST_CSV=True`로 변경해야 실행됩니다. 전체 train으로 XGB·LR을 학습하고, 각 시드별 g3를 전체 train으로 학습해 확률을 평균합니다. **test는 예측 입력으로만 사용**합니다. 현재 결과는 원본 10번 구성의 실험 후보이며 공식 11번 제출과 동일함을 확인하기 전에는 제출하지 마세요.

In [16]:
MAKE_TEST_CSV = False  # 검증 통과 및 팀 확인 후 True
if MAKE_TEST_CSV:
    if not PASS:
        raise RuntimeError('앙상블 개발 OOF 개선폭이 +0.003 미만이므로 test CSV 생성 중단')
    final_xgb = clone(xgb).fit(X_E, y, sample_weight=weight_array(y))
    final_lr = clone(pipeline_E_position_LR).fit(X_raw, y)
    if not np.array_equal(final_xgb.classes_, LABELS) or not np.array_equal(final_lr.classes_, LABELS):
        raise ValueError('클래스 순서 불일치')
    tx = checked(final_xgb.predict_proba(X_test_E), len(X_test_raw))
    tp = checked(final_lr.predict_proba(X_test_raw), len(X_test_raw))
    g3_test_probs = []
    for seed in SEEDS:
        model = clone(lgbm_g3)
        model.set_params(model__random_state=seed)
        model.fit(X_raw, y)
        if not np.array_equal(model.classes_, LABELS):
            raise ValueError('g3 클래스 순서 불일치')
        g3_test_probs.append(checked(model.predict_proba(X_test_raw),len(X_test_raw)))
    tg = checked(np.mean(g3_test_probs,axis=0),len(X_test_raw))
    final_p = checked((tx+tp+tg)/3,len(X_test_raw))
    # 제출 양식은 팀의 sample_submission.csv를 그대로 사용
    candidates = [DATA_DIR/'sample_submission.csv',ROOT/'sample_submission.csv']
    template = next((p for p in candidates if p.is_file()),None)
    if template is None:
        raise FileNotFoundError('sample_submission.csv를 확인해야 제출 파일을 생성할 수 있습니다')
    sub = pd.read_csv(template)
    if len(sub) != len(final_p):
        raise ValueError('제출 행 수가 예측 행 수와 다릅니다')
    target_cols = [c for c in sub.columns if c.lower() not in ('id','sample_id')]
    if len(target_cols)!=1:
        raise ValueError(f'제출 타깃 열을 자동 판별할 수 없습니다: {sub.columns.tolist()}')
    sub[target_cols[0]] = le.inverse_transform(final_p.argmax(1))
    path=OUT/'g3_seed_average_candidate.csv'
    if path.exists():
        raise FileExistsError(f'기존 결과 덮어쓰기 방지: {path}')
    sub.to_csv(path,index=False,encoding='utf-8-sig')
    print('후보 CSV 생성:',path)
else:
    print('Test 예측 비활성화. OOF 결과만 저장했습니다.')


Test 예측 비활성화. OOF 결과만 저장했습니다.
